# Aircraft Maintenance Data Pipeline and Feature Engineering

Files for the team:
- `labels.csv` for everyone
- `tabular_features.csv` for See Lin (XGBoost, unscaled)
- `tabular_features_scaled.csv` scaled version (optional)
- `sequences.npy` for Tanya / Linh / Helen (TS models, scaled float16)
- `seq_indices.npy`  Master Index alignment for sequences
- `channel_scaler.pkl` + `tabular_scaler.pkl`  for inference

NEW COLUMNS REFERENCE

labels.csv
rul_2d            — PRIMARY TARGET (binary)
                      0 = at risk: before_after=="before" AND date_diff >= -2
                          (flight within 2 days before maintenance)
                      1 = safe:    before_after=="after"
                          OR (before_after=="before" AND date_diff < -2)
                          (post-maintenance OR more than 2 days before)
            

rul_5d  and rul_10d ->same logic as 2d, to use if we expand our models

label_grouped     — top-10 maintenance labels kept as-is,
                      all remaining rare labels collapsed to "other"
                      (stretch goal: 11-class classification)

split             — train / val / test
                      70% train / 15% val / 15% test
                      stratified on rul_2d to preserve class balance
                      across all three splits

tabular_features.csv

All sensor feature columns follow naming: {channel}__{stat}

KEEP_CHANNELS -> redundant channels dropped based on EDA
(25 pairs with |r| > 0.9 → kept one representative per group):
  volt1             OAT
  amp1              IAS
  FQtyL             VSpd
  FQtyR             NormAc
  E1 FFlow          AltMSL
  E1 OilT
  E1 OilP
  E1 RPM
  E1 CHT1           (CHT2/3/4 dropped: r > 0.983 with CHT1)
  E1 EGT1           (EGT2/3/4 dropped: r > 0.949 with EGT1)

DROPPED CHANNELS:
  volt2             (r = 0.937 with volt1)
  amp2              (r > 0.9 with amp1, high missingness)
  E1 CHT2/3/4       (r > 0.983 with E1 CHT1)
  E1 EGT2/3/4       (r > 0.949 with E1 EGT1)

Per-channel stats :
  {ch}__mean        — arithmetic mean across all timesteps
  {ch}__miss_flag   — 1 if channel was entirely missing for this flight
                      (filled to 0 by imputation pipeline), 0 otherwise
etc...

Flight phase proportions:
  phase__ground_pct   — fraction of timesteps with E1 RPM <= 1000
                        (engine at ground/taxi RPM)
  phase__climb_pct    — fraction with RPM > 1000 AND VSpd > +100 fpm
  phase__cruise_pct   — fraction with RPM > 1000 AND -100 <= VSpd <= +100 fpm
  phase__descent_pct  — fraction with RPM > 1000 AND VSpd < -100 fpm

Cross-channel physics features:
  cross__fuel_efficiency  — mean(E1 FFlow / E1 RPM) where RPM > 800
                            fuel consumed per revolution
                            high values may indicate rich mixture or
                            dirty fuel injectors
  cross__oil_stress       — mean(E1 OilT / E1 OilP) where OilP > 10
                            oil temperature-to-pressure ratio
                            high values indicate stressed lubrication
  cross__cht_egt_ratio    — mean(E1 CHT1 / E1 EGT1) where EGT1 > 100
                            cylinder head to exhaust gas temp ratio
                            proxy for combustion efficiency
  cross__total_fuel       — mean(FQtyL + FQtyR) across flight
                            mean total fuel quantity
  cross__fuel_imbalance   — mean(|FQtyL - FQtyR|) across flight
                            asymmetric fuel burn between tanks
  cross__volt_drop        — mean(volt1) - min(volt1)
                            voltage drop range across flight
                            high values may indicate alternator or
                            battery degradation
  cross__elec_load        — mean(amp1)
                            mean electrical current draw

Flight metadata:
  meta__flight_length     — number of timesteps in this flight
                            (same as flight_length in labels.csv)


sequences.npy

Shape: (N_flights, 9212, 15)  dtype: float16

Axis 0 — flights (aligned to seq_indices.npy)
Axis 1 — timesteps
          fixed length = 9212 (p95 from EDA)
          flights longer than 9212: last 9212 steps kept (most recent)
          flights shorter than 9212: left-padded with 0
Axis 2 — channels (in this exact order):
          0: volt1      1: amp1       2: FQtyL      3: FQtyR
          4: E1 FFlow   5: E1 OilT    6: E1 OilP   7: E1 RPM
          8: E1 CHT1    9: E1 EGT1   10: OAT      11: IAS
          12: VSpd      13: NormAc   14: AltMSL

Preprocessing applied:
  - NaN imputed per flight (ffill → bfill → 0 for full-channel missing)
  - RobustScaler fitted on training flights only (no leakage)
  - Encoded as float16 to reduce file size (~6 GB vs ~12 GB in float32)

seq_indices.npy

Shape: (N_flights,)  dtype: int64
Master Index value for each row in sequences.npy
Use to align sequences with labels.csv:
  y     = labels.loc[seq_indices, 'rul_2d'].values
  split = labels.loc[seq_indices, 'split'].values
  

In [1]:
#Imports & Configuration

import os, gc, warnings, pickle
import numpy as np
import pandas as pd
import dask.dataframe as dd
from numpy.lib.format import open_memmap
from scipy.stats import skew as sp_skew, kurtosis as sp_kurtosis
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)

# Paths
FULL_DATA_DIR = '/kaggle/input/datasets/hooong/aviation-maintenance-dataset-from-the-ngafid/all_flights/all_flights'
OUT_DIR       = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

ALL_CHANNELS = [
    'volt1','volt2','amp1','amp2','FQtyL','FQtyR',
    'E1 FFlow','E1 OilT','E1 OilP','E1 RPM',
    'E1 CHT1','E1 CHT2','E1 CHT3','E1 CHT4',
    'E1 EGT1','E1 EGT2','E1 EGT3','E1 EGT4',
    'OAT','IAS','VSpd','NormAc','AltMSL'
]

# Drop redundant channels
# Keep one representative per correlated group
DROP_CHANNELS = ['volt2','amp2','E1 CHT2','E1 CHT3','E1 CHT4','E1 EGT2','E1 EGT3','E1 EGT4']
KEEP_CHANNELS = [c for c in ALL_CHANNELS if c not in DROP_CHANNELS]
# Result: 15 channels

# Pipeline settings
SEQ_LEN      = 9212  
SHORT_CUTOFF = 100  
TAU_VALUES   = [2, 5, 10] 
RANDOM_SEED  = 42

print(f'Keep channels ({len(KEEP_CHANNELS)}): {KEEP_CHANNELS}')
print(f'Sequence length (p95): {SEQ_LEN}')
print(f'RUL thresholds (days): {TAU_VALUES}')

Keep channels (15): ['volt1', 'amp1', 'FQtyL', 'FQtyR', 'E1 FFlow', 'E1 OilT', 'E1 OilP', 'E1 RPM', 'E1 CHT1', 'E1 EGT1', 'OAT', 'IAS', 'VSpd', 'NormAc', 'AltMSL']
Sequence length (p95): 9212
RUL thresholds (days): [2, 5, 10]


In [2]:
# Load header, filter, derive labels, split

flight_header_df = pd.read_csv(
    os.path.join(FULL_DATA_DIR, 'flight_header.csv'), index_col='Master Index'
)
flight_data_df = dd.read_parquet(os.path.join(FULL_DATA_DIR, 'one_parq'))
print(f'Raw flights       : {len(flight_header_df):,}')
print(f'Parquet partitions: {flight_data_df.npartitions}')

#Remove short flights
valid_header = flight_header_df[flight_header_df['flight_length'] >= SHORT_CUTOFF].copy()
print(f'\nAfter short-flight filter : {len(valid_header):,}  '
      f'(removed {len(flight_header_df)-len(valid_header):,} flights, '
      f'{(1-len(valid_header)/len(flight_header_df))*100:.1f}%)')

#Exclude maintenance-day flights
labeled = valid_header[valid_header['before_after'] != 'same'].copy()
print(f'After excluding same     : {len(labeled):,}  '
      f'(removed {len(valid_header)-len(labeled):,} flights)')

# Step 3: Derive binary RUL targets at each tau
# rul = 1 (SAFE): flight is after maintenance OR before but > t days away
# rul = 0 (RISK): flight is before AND within t days of maintenance
for tau in TAU_VALUES:
    labeled[f'rul_{tau}d'] = (
        (labeled['before_after'] == 'after') |
        (labeled['date_diff'] < -tau)
    ).astype(int)

print('\nRUL class balance:')
for tau in TAU_VALUES:
    vc = labeled[f'rul_{tau}d'].value_counts().sort_index()
    n  = len(labeled)
    print(f'  tau={tau:2d}d -> risk(0): {vc.get(0,0):,} ({vc.get(0,0)/n*100:.1f}%)  '
          f'safe(1): {vc.get(1,0):,} ({vc.get(1,0)/n*100:.1f}%)')

#Top-10 label grouping for stretch goal
top10 = labeled['label'].value_counts().nlargest(10).index
labeled['label_grouped'] = labeled['label'].where(labeled['label'].isin(top10), other='other')
print(f'\nTop-10 labels cover {labeled["label"].isin(top10).mean()*100:.1f}% of flights')

#Stratified 70/15/15 split on rul_2d
train_idx, temp_idx = train_test_split(
    labeled.index, test_size=0.30,
    stratify=labeled['rul_2d'], random_state=RANDOM_SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50,
    stratify=labeled.loc[temp_idx, 'rul_2d'], random_state=RANDOM_SEED
)
labeled['split'] = 'train'
labeled.loc[val_idx,  'split'] = 'val'
labeled.loc[test_idx, 'split'] = 'test'

print('\nSplit summary:')
for sp in ['train','val','test']:
    sub = labeled[labeled['split']==sp]
    vc  = sub['rul_2d'].value_counts(normalize=True)
    print(f'  {sp:5s}: {len(sub):,} | class0: {vc.get(0,0)*100:.1f}%  class1: {vc.get(1,0)*100:.1f}%')

labeled[['before_after','date_diff','flight_length','label',
         'label_grouped','rul_2d','rul_5d','rul_10d','split']].to_csv(f'{OUT_DIR}/labels.csv')
print(f'\nSaved: labels.csv  ({len(labeled):,} flights)')

Raw flights       : 28,935
Parquet partitions: 401

After short-flight filter : 24,934  (removed 4,001 flights, 13.8%)
After excluding same     : 19,540  (removed 5,394 flights)

RUL class balance:
  tau= 2d -> risk(0): 7,889 (40.4%)  safe(1): 11,651 (59.6%)
  tau= 5d -> risk(0): 10,119 (51.8%)  safe(1): 9,421 (48.2%)
  tau=10d -> risk(0): 10,464 (53.6%)  safe(1): 9,076 (46.4%)

Top-10 labels cover 79.3% of flights

Split summary:
  train: 13,678 | class0: 40.4%  class1: 59.6%
  val  : 2,931 | class0: 40.4%  class1: 59.6%
  test : 2,931 | class0: 40.4%  class1: 59.6%

Saved: labels.csv  (19,540 flights)


In [3]:
# Fit RobustScaler on training sample

train_idx_set = set(train_idx)

def sample_train_partition(df):
    df = df[df.index.isin(train_idx_set)]
    if len(df) == 0:
        return pd.DataFrame(columns=KEEP_CHANNELS)
    df = df[[c for c in KEEP_CHANNELS if c in df.columns]].fillna(0)
    return df.sample(n=min(200, len(df)), random_state=42)

print('Fitting RobustScaler on training data sample...')
scaler_sample = flight_data_df.map_partitions(sample_train_partition).compute()
scaler_sample = scaler_sample[[c for c in KEEP_CHANNELS if c in scaler_sample.columns]]

channel_scaler = RobustScaler()
channel_scaler.fit(scaler_sample)

del scaler_sample; gc.collect()

with open(f'{OUT_DIR}/channel_scaler.pkl', 'wb') as f:
    pickle.dump(channel_scaler, f)
print('Scaler fitted. Saved: channel_scaler.pkl')

Fitting RobustScaler on training data sample...
Scaler fitted. Saved: channel_scaler.pkl


In [4]:
# Per-flight feature extraction function

STAT_NAMES = ['mean','std','min','max','range','p05','p25','p50','p75','p95',
              'iqr','slope','first','last','delta','skew','kurt','cv','miss_flag']

def extract_flight_features(group):
    out = {}
    T   = len(group)

    # Per-channel descriptive statistics
    for ch in KEEP_CHANNELS:
        pfx = ch + '__'

        if ch not in group.columns:
            for s in STAT_NAMES: out[pfx+s] = np.nan
            continue

        raw = group[ch].values.astype(np.float64)

        # miss_flag: channel was entirely missing 
        is_missing = (raw.std() < 1e-4) and (abs(raw.mean()) < 1e-4)
        out[pfx+'miss_flag'] = int(is_missing)

        if is_missing or len(raw) < 3:
            for s in STAT_NAMES[:-1]: out[pfx+s] = np.nan
            continue


        out[pfx+'mean']  = raw.mean()
        out[pfx+'std']   = raw.std()
        out[pfx+'min']   = raw.min()
        out[pfx+'max']   = raw.max()
        out[pfx+'range'] = raw.max() - raw.min()


        p05, p25, p50, p75, p95 = np.percentile(raw, [5, 25, 50, 75, 95])
        out[pfx+'p05'] = p05; out[pfx+'p25'] = p25; out[pfx+'p50'] = p50
        out[pfx+'p75'] = p75; out[pfx+'p95'] = p95; out[pfx+'iqr'] = p75 - p25

        x = np.arange(len(raw))
        out[pfx+'slope'] = float(np.polyfit(x, raw, 1)[0])  # linear drift across flight
        out[pfx+'first'] = float(raw[0])    # value at flight start
        out[pfx+'last']  = float(raw[-1])   # value at flight end
        out[pfx+'delta'] = float(raw[-1] - raw[0])  # total drift

        out[pfx+'skew'] = float(sp_skew(raw))
        out[pfx+'kurt'] = float(sp_kurtosis(raw))
        out[pfx+'cv']   = float(raw.std() / (abs(raw.mean()) + 1e-9))

    # Flight phase proportions 
    if 'E1 RPM' in group.columns and 'VSpd' in group.columns:
        rpm  = group['E1 RPM'].values.astype(np.float64)
        vspd = group['VSpd'].values.astype(np.float64)
        flying  = rpm > 1000          
        ground  = ~flying
        climb   = flying & (vspd >  100)   
        descent = flying & (vspd < -100)   
        cruise  = flying & ~climb & ~descent
        out['phase__ground_pct']  = float(ground.mean())
        out['phase__climb_pct']   = float(climb.mean())
        out['phase__cruise_pct']  = float(cruise.mean())
        out['phase__descent_pct'] = float(descent.mean())
    else:
        for k in ['phase__ground_pct','phase__climb_pct',
                  'phase__cruise_pct','phase__descent_pct']: out[k] = np.nan

    # Cross-channel physics features
    def safe_ratio(n_ch, d_ch, d_min=0):
        if n_ch not in group.columns or d_ch not in group.columns: return np.nan
        n = group[n_ch].values.astype(np.float64)
        d = group[d_ch].values.astype(np.float64)
        ok = d > d_min
        return float((n[ok]/(d[ok]+1e-9)).mean()) if ok.sum() > 0 else np.nan

    out['cross__fuel_efficiency']  = safe_ratio('E1 FFlow', 'E1 RPM', d_min=800)
    out['cross__oil_stress']       = safe_ratio('E1 OilT', 'E1 OilP', d_min=10)
    out['cross__cht_egt_ratio']    = safe_ratio('E1 CHT1', 'E1 EGT1', d_min=100)

    if 'FQtyL' in group.columns and 'FQtyR' in group.columns:
        l = group['FQtyL'].values.astype(np.float64)
        r = group['FQtyR'].values.astype(np.float64)
        out['cross__total_fuel']     = float((l+r).mean())   # total fuel quantity
        out['cross__fuel_imbalance'] = float(abs(l-r).mean())  # asymmetric burn
    else:
        out['cross__total_fuel'] = out['cross__fuel_imbalance'] = np.nan

    if 'volt1' in group.columns:
        v = group['volt1'].values.astype(np.float64)
        out['cross__volt_drop'] = float(v.mean() - v.min())
    else:
        out['cross__volt_drop'] = np.nan

    if 'amp1' in group.columns:
        out['cross__elec_load'] = float(group['amp1'].values.astype(np.float64).mean())
    else:
        out['cross__elec_load'] = np.nan

    #Metadata
    out['meta__flight_length'] = T

    return out

# Smoke test
_test = pd.DataFrame(np.random.randn(100, len(KEEP_CHANNELS)), columns=KEEP_CHANNELS)
_f    = extract_flight_features(_test)
print(f'Features per flight  : {len(_f)}')
print(f'  Sensor stats       : {len([k for k in _f if "__" in k and not k.startswith(("phase","cross","meta"))])}')
print(f'  Phase features     : {len([k for k in _f if k.startswith("phase")])}')
print(f'  Cross-channel      : {len([k for k in _f if k.startswith("cross")])}')
print(f'  Metadata           : {len([k for k in _f if k.startswith("meta")])}')
del _test, _f

Features per flight  : 297
  Sensor stats       : 285
  Phase features     : 4
  Cross-channel      : 7
  Metadata           : 1


In [5]:
# Main processing loop

labeled_set  = set(labeled.index)
valid_list   = sorted(labeled.index)
n_flights    = len(valid_list)
flight_pos   = {idx: i for i, idx in enumerate(valid_list)}
n_channels   = len(KEEP_CHANNELS)

seq_path = f'{OUT_DIR}/sequences.npy'
seq_mmap = open_memmap(
    seq_path, mode='w+', dtype=np.float16,
    shape=(n_flights, SEQ_LEN, n_channels)
)

seq_written    = np.zeros(n_flights, dtype=bool)
seq_indices    = np.array(valid_list)
tabular_records = {}

print(f'Flights to process : {n_flights:,}')
print(f'Sequence shape     : ({n_flights}, {SEQ_LEN}, {n_channels})')
print(f'Seq file on disk   : ~{n_flights*SEQ_LEN*n_channels*2/1e9:.1f} GB')
print()

for part_i in tqdm(range(flight_data_df.npartitions), desc='Partitions'):

    # Load one partition
    part = flight_data_df.get_partition(part_i).compute()

    part = part[part.index.isin(labeled_set)]
    if len(part) == 0:
        del part; gc.collect(); continue

    # Sort timesteps
    if 'timestep' in part.columns:
        part = part.sort_values('timestep')

    # Impute per flight 
    sensor_cols = [c for c in ALL_CHANNELS if c in part.columns]
    part[sensor_cols] = (
        part.groupby(level=0)[sensor_cols]
        .transform(lambda x: x.ffill().bfill())
    )
    part[sensor_cols] = part[sensor_cols].fillna(0)  # remaining = full-channel NaN

    # Process each flight
    for idx, group in part.groupby(level=0):
        if idx not in flight_pos: continue
        pos = flight_pos[idx]

        tabular_records[idx] = extract_flight_features(group)

        keep_cols = [c for c in KEEP_CHANNELS if c in group.columns]
        raw_arr   = group[keep_cols].values.astype(np.float32)

        # Handle missing channels
        if len(keep_cols) < n_channels:
            full_arr = np.zeros((len(raw_arr), n_channels), dtype=np.float32)
            for ci, ch in enumerate(KEEP_CHANNELS):
                if ch in keep_cols:
                    full_arr[:, ci] = raw_arr[:, keep_cols.index(ch)]
            raw_arr = full_arr

        scaled = channel_scaler.transform(raw_arr).astype(np.float16)

        T = scaled.shape[0]
        if T >= SEQ_LEN:
            seq_mmap[pos] = scaled[-SEQ_LEN:]
        else:
            seq_mmap[pos, SEQ_LEN-T:, :] = scaled

        seq_written[pos] = True

    del part; gc.collect()

del seq_mmap; gc.collect()

print(f'\nFlights processed  : {seq_written.sum():,} / {n_flights:,}')
if (~seq_written).sum() > 0:
    print(f'WARNING: {(~seq_written).sum()} flights in labels not found in parquet')
print(f'Tabular records    : {len(tabular_records):,}')

Flights to process : 19,540
Sequence shape     : (19540, 9212, 15)
Seq file on disk   : ~5.4 GB



Partitions:   0%|          | 0/401 [00:00<?, ?it/s]


Flights processed  : 19,540 / 19,540
Tabular records    : 19,540


In [6]:
#Assemble tabular features

tabular_df = pd.DataFrame.from_dict(tabular_records, orient='index')
tabular_df.index.name = 'Master Index'

tabular_full = tabular_df.join(
    labeled[['rul_2d','rul_5d','rul_10d','label','label_grouped','split']],
    how='inner'
)

feat_cols = [c for c in tabular_df.columns]
nan_pct   = tabular_full[feat_cols].isna().mean().mean() * 100
print(f'Feature columns  : {len(feat_cols)}')
print(f'Overall NaN rate : {nan_pct:.1f}%')
if nan_pct > 10:
    top_nan = tabular_full[feat_cols].isna().mean().sort_values(ascending=False).head(5)
    print('Top NaN features:'); print(top_nan[top_nan>0])

tabular_full.to_csv(f'{OUT_DIR}/tabular_features.csv')
print(f'\nSaved: tabular_features.csv  {tabular_full.shape}')

train_mask = tabular_full['split'] == 'train'
tabular_scaler = RobustScaler()
tabular_scaler.fit(tabular_full.loc[train_mask, feat_cols].fillna(0))

tabular_scaled = tabular_full.copy()
tabular_scaled[feat_cols] = tabular_scaler.transform(tabular_full[feat_cols].fillna(0))
tabular_scaled.to_csv(f'{OUT_DIR}/tabular_features_scaled.csv')

with open(f'{OUT_DIR}/tabular_scaler.pkl', 'wb') as f:
    pickle.dump(tabular_scaler, f)

print(f'Saved: tabular_features_scaled.csv')
print(f'Saved: tabular_scaler.pkl')

del tabular_df, tabular_records; gc.collect()

Feature columns  : 297
Overall NaN rate : 1.2%

Saved: tabular_features.csv  (19540, 303)
Saved: tabular_features_scaled.csv
Saved: tabular_scaler.pkl


0

In [7]:
# Trim sequence index

seq_indices_final = seq_indices[seq_written]
np.save(f'{OUT_DIR}/seq_indices.npy', seq_indices_final)

seq_v  = np.load(seq_path, mmap_mode='r')
sidx_v = np.load(f'{OUT_DIR}/seq_indices.npy')
print(f'sequences.npy shape : {seq_v.shape}  dtype={seq_v.dtype}')
print(f'seq_indices.npy     : {len(sidx_v):,} entries')
del seq_v
print('Saved: sequences.npy  seq_indices.npy')

sequences.npy shape : (19540, 9212, 15)  dtype=float16
seq_indices.npy     : 19,540 entries
Saved: sequences.npy  seq_indices.npy


In [8]:
#Quality checks

print('='*55)
print('QUALITY CHECKS')
print('='*55)

ldf = pd.read_csv(f'{OUT_DIR}/labels.csv', index_col='Master Index')
print(f'\n[labels.csv]  {ldf.shape}')
print(f'  Splits: {ldf["split"].value_counts().to_dict()}')
for tau in TAU_VALUES:
    vc = ldf[f'rul_{tau}d'].value_counts().sort_index()
    print(f'  rul_{tau}d: class0={vc.get(0,0):,}  class1={vc.get(1,0):,}')
assert ldf['split'].nunique() == 3
assert ldf['rul_2d'].isna().sum() == 0
print('  OK')

tdf  = pd.read_csv(f'{OUT_DIR}/tabular_features.csv', index_col='Master Index')
fcols = [c for c in tdf.columns if '__' in c]
print(f'\n[tabular_features.csv]  {tdf.shape}')
print(f'  Feature columns : {len(fcols)}')
print(f'  NaN rate        : {tdf[fcols].isna().mean().mean()*100:.1f}%')
print(f'  Train rows      : {(tdf["split"]=="train").sum():,}')
assert len(tdf) > 0 and 'rul_2d' in tdf.columns
print('  OK')

seq  = np.load(seq_path, mmap_mode='r')
sidx = np.load(f'{OUT_DIR}/seq_indices.npy')
print(f'\n[sequences.npy]  {seq.shape}  dtype={seq.dtype}')
print(f'  Non-zero rows   : {(seq.reshape(seq.shape[0],-1).sum(axis=1)!=0).sum():,}')
assert seq.shape[0] == len(sidx)
assert seq.shape[2] == len(KEEP_CHANNELS)
print('  OK')

common = len(set(sidx) & set(ldf.index))
print(f'\n[Alignment]  {common:,}/{len(sidx):,} sequences have labels')
assert common > 0.95 * len(sidx)
print('  OK')

del seq, tdf, ldf; gc.collect()
print('\nAll checks passed')

QUALITY CHECKS

[labels.csv]  (19540, 9)
  Splits: {'train': 13678, 'val': 2931, 'test': 2931}
  rul_2d: class0=7,889  class1=11,651
  rul_5d: class0=10,119  class1=9,421
  rul_10d: class0=10,464  class1=9,076
  OK

[tabular_features.csv]  (19540, 303)
  Feature columns : 297
  NaN rate        : 1.2%
  Train rows      : 13,678
  OK

[sequences.npy]  (19540, 9212, 15)  dtype=float16
  Non-zero rows   : 19,540
  OK

[Alignment]  19,540/19,540 sequences have labels
  OK

All checks passed


In [9]:
#Output summary

print('='*55)
print('PIPELINE COMPLETE — files in /kaggle/working/')
print('='*55)
for fname in sorted(os.listdir(OUT_DIR)):
    fsize = os.path.getsize(f'{OUT_DIR}/{fname}')
    unit  = 'GB' if fsize > 1e9 else 'MB'
    size  = fsize/1e9 if fsize > 1e9 else fsize/1e6
    print(f'  {fname:<45}  {size:6.1f} {unit}')

print('''
TEAMMATE GUIDE

EVERYONE: load labels first:
  import pandas as pd, numpy as np
  BASE   = "/kaggle/input/aircraft-pipeline"
  labels = pd.read_csv(f"{BASE}/labels.csv", index_col="Master Index")

SEE LIN (XGBoost):
  tab   = pd.read_csv(f"{BASE}/tabular_features.csv", index_col="Master Index")
  feats = [c for c in tab.columns if "__" in c]
  X_train = tab[tab["split"]=="train"][feats].fillna(0)
  y_train = tab[tab["split"]=="train"]["rul_2d"].values
  X_val   = tab[tab["split"]=="val"][feats].fillna(0)
  y_val   = tab[tab["split"]=="val"]["rul_2d"].values
  X_test  = tab[tab["split"]=="test"][feats].fillna(0)
  y_test  = tab[tab["split"]=="test"]["rul_2d"].values
  # Stretch goal: swap rul_2d for label_grouped (10-class + other)

TANYA / LINH / HELEN (TS models):
  X   = np.load(f"{BASE}/sequences.npy", mmap_mode="r")  # (N, 9212, 15)
  idx = np.load(f"{BASE}/seq_indices.npy")
  y     = labels.loc[idx, "rul_2d"].values   # swap for rul_5d / rul_10d
  split = labels.loc[idx, "split"].values
  X_train, y_train = X[split=="train"], y[split=="train"]
  X_val,   y_val   = X[split=="val"],   y[split=="val"]
  X_test,  y_test  = X[split=="test"],  y[split=="test"]

CHANNEL ORDER in sequences (axis=2):
  0:volt1  1:amp1  2:FQtyL  3:FQtyR  4:E1 FFlow
  5:E1 OilT  6:E1 OilP  7:E1 RPM  8:E1 CHT1  9:E1 EGT1
  10:OAT  11:IAS  12:VSpd  13:NormAc  14:AltMSL
  Already: RobustScaled (train-fitted), left-padded 0, float16

FOR SENSITIVITY ANALYSIS (Linh):
  y = labels.loc[idx, "rul_2d"].values   # tau=2 (paper baseline)
  y = labels.loc[idx, "rul_5d"].values   # tau=5
  y = labels.loc[idx, "rul_10d"].values  # tau=10

LIMITATIONS:
  - Group-aware split not implemented (flights from same maintenance
    event may span train/test). Flag in report as future work.
  - Flights spanning multiple parquet partitions: rare edge case,
    handled by fillna(0) at partition boundary.
''')

PIPELINE COMPLETE — files in /kaggle/working/
  __notebook__.ipynb                                0.0 MB
  channel_scaler.pkl                                0.0 MB
  labels.csv                                        1.7 MB
  seq_indices.npy                                   0.2 MB
  sequences.npy                                     5.4 GB
  tabular_features.csv                             68.3 MB
  tabular_features_scaled.csv                     105.7 MB
  tabular_scaler.pkl                                0.0 MB

TEAMMATE GUIDE

EVERYONE: load labels first:
  import pandas as pd, numpy as np
  BASE   = "/kaggle/input/aircraft-pipeline"
  labels = pd.read_csv(f"{BASE}/labels.csv", index_col="Master Index")

SEE LIN (XGBoost):
  tab   = pd.read_csv(f"{BASE}/tabular_features.csv", index_col="Master Index")
  feats = [c for c in tab.columns if "__" in c]
  X_train = tab[tab["split"]=="train"][feats].fillna(0)
  y_train = tab[tab["split"]=="train"]["rul_2d"].values
  X_val   = tab[tab["spli